[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/Rang/blob/main/tools/examole/02_extract_colors.ipynb)

<div style="font-family:Arial,sans-serif">

# 02. Extract colors with k-means

Run k-means separately in every saved region and review the candidate sheet.

Created by **Mohsen Tahmasebi Nasab, PhD**<br>
[hydromohsen.com](https://hydromohsen.com)

Copyright and license holder: Mohsen Tahmasebi Nasab. Notebook code is
licensed under the repository's MIT License. Rang palette data follows the
CC0 dedication described in the licensing guide. Source images keep their own
rights and reuse terms.

</div>

<div style="font-family:Arial,sans-serif;background:#fff3cd;padding:14px"><strong>YOUR INPUT</strong><br>Set the Git branch, choose whether to use Google Drive, and give the palette a short filename.</div>

In [ ]:
REPO_REF = "main" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
PALETTE_SLUG = "kashan" #@param {type:"string"}

Google Drive keeps the recipe available when you move to the next notebook.
For a local Jupyter session, working files are placed under
`cache/notebook_workflow/`. When testing a GitHub branch, replace `main` with
the branch name above.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_ROOT = Path("/content/Rang")
    if not (REPO_ROOT / "tools" / "notebook_workflow.py").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", REPO_REF,
            "https://github.com/mohsennasab/Rang.git", str(REPO_ROOT)
        ], check=True)
else:
    probe = Path.cwd().resolve()
    REPO_ROOT = next(
        candidate for candidate in (probe, *probe.parents)
        if (candidate / "tools" / "notebook_workflow.py").exists()
    )

try:
    import matplotlib
    import numpy
    import PIL
    import sklearn
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(REPO_ROOT / "tools" / "requirements.txt")
    ], check=True)

sys.path.insert(0, str(REPO_ROOT / "tools"))

from notebook_workflow import *

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Rang") / PALETTE_SLUG
else:
    WORK_DIR = REPO_ROOT / "cache" / "notebook_workflow" / PALETTE_SLUG

WORK_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_PATH = WORK_DIR / f"{PALETTE_SLUG}-recipe.json"
use_arial()
print("Repository:", REPO_ROOT)
print("Working folder:", WORK_DIR)
print("Recipe:", RECIPE_PATH)

In [ ]:
if not RECIPE_PATH.exists():
    raise FileNotFoundError(
        f"Recipe not found at {RECIPE_PATH}. Run notebook 01 first and use "
        "the same Google Drive and palette slug settings."
    )
recipe = read_json(RECIPE_PATH)
print(f'Loaded {recipe["palette"]} with {len(recipe["regions"])} regions')

In [ ]:
region_overlay(RECIPE_PATH, WORK_DIR, WORK_DIR / "regions.png")

Each region is clustered in CIELAB. The candidate sheet orders clusters by
their share within that region. A large region does not get more authority
than a small region. The rows are evidence for your decision, not a finished
palette.

<div style="font-family:Arial,sans-serif;background:#d9edf7;padding:14px"><strong>YOUR DECISION</strong><br>Choose whether this extraction run should become the accepted candidate snapshot in the recipe.</div>

In [ ]:
ACCEPT_THIS_RUN = True #@param {type:"boolean"}

In [ ]:
candidates = save_candidates(
    RECIPE_PATH, WORK_DIR, accept=ACCEPT_THIS_RUN
)
print(f"Extracted {len(candidates)} candidates")
for candidate in candidates:
    print(candidate["id"], candidate["hex"],
          f'{candidate["share"]:.1f}%')
print("Candidate sheet:", WORK_DIR / "candidates.png")